<a href="https://colab.research.google.com/github/Garimasharma8/Learning_transformers/blob/main/A_full_training_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A full training Loop

A full training loop consists of data --> model --> evaluate --> gradient --> optimization --> model.

In this notebook we will create the full training loop as mentioned above in PyTorch using best practices.

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


## Step 1: Load data

In this step we will load the raw mrpc data, prepare for tokenization using map function, tokenize the data, and make sure to use datacollator for batching

In [2]:
from transformers import AutoTokenizer, DataCollatorWithPadding

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_datasets


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# Step 2: Prepare data for training
Now the tokenized dataset contains the input ids, which are actually the tokens needed to train the model. But here we have some redundant information that can be deleted such as sentence 1 and sentence 2, and idx. Set the format of the datasets so they return PyTorch tensors instead of lists.

In [3]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1","sentence2","idx"])
tokenized_datasets = tokenized_datasets.rename_column("label","labels")
tokenized_datasets.set_format("torch")
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

Now that this is done, we can easily define our dataloaders, dataloaders are responsible to convert our dataset into batches. The dataloaders are present in "torch utilities for data". The dataloaders need a collate function that points to our data collator. In this code, we will convert our training and validation dataset to training and validation dataloader.

In [4]:
from torch.utils.data import DataLoader

train_dataLoader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator)
val_dataLoader = DataLoader(tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator)



To quickly check our batches, pick up the first batch from the train_dataLoader and check the shape of all columns.

In [5]:
for batch in train_dataLoader:
  break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 76]),
 'token_type_ids': torch.Size([8, 76]),
 'attention_mask': torch.Size([8, 76])}

Now we have batches of size 8 ready.

# Step 3: Load a suitable model
we will chose a suitable model from transformers library using AutoModelforSequenceClassification and use the same checkpoint as we have used for tokenizer.

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


To quickly check that the model is predicting fine, we pass our first batch as the test data

In [7]:
outputs = model(**batch)
outputs


SequenceClassifierOutput(loss=tensor(0.6965, grad_fn=<NllLossBackward0>), logits=tensor([[-0.2896, -0.3060],
        [-0.2988, -0.3040],
        [-0.3065, -0.2997],
        [-0.2916, -0.2923],
        [-0.2958, -0.2948],
        [-0.2854, -0.2959],
        [-0.2887, -0.3038],
        [-0.2869, -0.3165]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

for the first 8 sequences, we got the logits and output.

In [8]:
print(outputs.loss, outputs.logits.shape)

tensor(0.6965, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


# Step 4: Select a suitable optimizer and learning schedular
Now we are ready for the training loop the only thing we need is an optimizer and leaning schedular. Since we are trying to replicate the trainer API here that uses AdamW (adam with weight decay regularization), we will use the same optimizer here AdamW. The optimizers are imported from optim module of torch library.

In [9]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 5e-05
    maximize: False
    weight_decay: 0.01
)

Modern Optimization Tips: For even better performance, you can try:

- AdamW with weight decay: AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
- 8-bit Adam: Use bitsandbytes for memory-efficient optimization
- Different learning rates: Lower learning rates (1e-5 to 3e-5) often work better for large models

Finally, the learning rate scheduler used by default is just a linear decay from the maximum value (5e-5) to 0. To properly define it, we need to know the number of training steps we will take, which is the number of epochs we want to run multiplied by the number of training batches (which is the length of our training dataloader). The Trainer uses three epochs by default, so we will follow that:

In [10]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataLoader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

1377


# Step 5: Training Loop
We are now ready to train! To get some sense of when training will be finished, we add a progress bar over our number of training steps, using the tqdm library:

In [11]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()

for epoch in range(num_epochs):
  for batch in train_dataLoader:
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()

    optimizer.step()
    lr_scheduler.step()
    optimizer.zero_grad()
    progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

Modern Training Optimizations: To make your training loop even more efficient, consider:

- Gradient Clipping: Add torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) before optimizer.step()
- Mixed Precision: Use torch.cuda.amp.autocast() and GradScaler for faster training
- Gradient Accumulation: Accumulate gradients over multiple batches to simulate larger batch sizes
- Checkpointing: Save model checkpoints periodically to resume training if interrupted

# Step 6: Evaluation loop
As we did earlier, we will use a metric provided by the 🤗 Evaluate library. We’ve already seen the metric.compute() method, but metrics can actually accumulate batches for us as we go over the prediction loop with the method add_batch(). Once we have accumulated all the batches, we can get the final result with metric.compute(). Here’s how to implement all of this in an evaluation loop:

In [13]:
import evaluate, torch

metric = evaluate.load("glue", "mrpc")
# eval() mode disables dropout and uses running statistics for batch norm instead of computing them from the current batch.
model.eval()
for batch in val_dataLoader:
  # torch.nograd() To save memory and speed up computation by disabling gradient tracking.
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()


{'accuracy': 0.8480392156862745, 'f1': 0.8945578231292517}

# Step 7: Accelerate the training using accelerator (Optional)
As of now, we are training on CPU or GPU, but we can also train on multiple CPU and multiple GPUs. Trainer API handles this distribution of load automatically for us, but it then becomes the black box/grey box and we have less control over it. Using accelerator, we can still do the same custom training loop with distributed training. The full training loop would be:

In [18]:
from accelerate import Accelerator
import torch
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, val_dl, model, optimizer = accelerator.prepare(
    train_dataLoader, val_dataLoader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/1377 [00:00<?, ?it/s]

The big difference in training, without accelerator the training took 1 hour 26 minutes, with accelerator training time is reduced to 3  minutes only. Which is a big benefit of using accelerator

Putting this in a train.py script will make that script runnable on any kind of distributed setup. To try it out in your distributed setup, run the command:


accelerate config
which will prompt you to answer a few questions and dump your answers in a configuration file used by this command:


accelerate launch train.py
which will launch the distributed training.

If you want to try this in a Notebook (for instance, to test it with TPUs on Colab), just paste the code in a training_function() and run a last cell with:


from accelerate import notebook_launcher

notebook_launcher(training_function)

Key Takeaways:

- Manual training loops give you complete control but require understanding of the proper sequence: forward → backward → optimizer step → scheduler step → zero gradients
- AdamW with weight decay is the recommended optimizer for transformer models
- Always use model.eval() and torch.no_grad() during evaluation for correct behavior and efficiency
- Accelerate makes distributed training accessible with minimal code changes
Device management (moving tensors to GPU/CPU) is crucial for PyTorch operations
- Modern techniques like mixed precision, gradient accumulation, and gradient clipping can significantly improve training efficiency